In [11]:
TOPIC_COMMENT_PATH = r'..\data\fe\topic_comment.parquet'

# 🧹 数据清洗

## 清洗策略概述

基于数据探索，制定以下清洗策略：

### 1. 全局清洗
- **去重**：`user_weibo` 存在大量同用户重复 weibo_id（68,809 组），需去重
- **类型统一**：`user_info.user_id` 为 `object` 类型，其他表为 `int64`，需统一为 `int64` 以便关联

### 2. 文本清洗（保留情绪信号）
- **移除 HTML 标签**：`user_weibo` 中 972 条含 HTML 标签
- **移除 URL 链接**
- **清洗话题标签 `#xxx#`**：提取标签内容保留，去掉 `#` 符号（标签本身含情绪信息）
- **保留 `@用户` 引用**：体现社交互动关系，仅在分析文本时按需移除
- **保留表情符号 `[xxx]`**：微博表情是重要的情绪信号
- **处理 `回复@xxx：` 格式**：提取回复目标用户信息后清理前缀

### 3. 行为数据清洗
- **转发微博处理**：内容为"转发微博"的无文本信息价值，但保留 `reposted_weibo_id` 作为转发关系信号，添加 `is_repost` 标记列
- **空文本处理**：
  - `topic_comment`：6,517 条空评论 → 移除（无情绪信息）
  - `user_weibo`：9,302 条空微博 → 移除


### 4. 数据质量过滤
- **过滤极短评论**（<=2字符且无情绪价值的，如空字符串）
- **保留极短但有情绪表达的评论**（如"加油""唉""晚安"等，这些本身就是情绪表达）

## Step 1: 重新加载原始数据 & 去重 & 类型统一

In [12]:
import pandas as pd
import re

# ========== 加载原始数据 ==========
df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)


# 记录表原始行数，用于全程追踪保留率
_ORIGINAL_COUNTS = {
    "topic_comment": len(df_topic_comment),
}

def retention_rate(table: str, current_df) -> str:
    """计算相对于原始数据的保留率。"""
    orig = _ORIGINAL_COUNTS[table]
    cur = len(current_df)
    return f"{cur:,} / {orig:,} ({cur / orig * 100:.1f}% 保留)"

print("\n📦 原始数据量:")
print(f"  topic_comment: {len(df_topic_comment):>10,}")


📦 原始数据量:
  topic_comment:    121,807


## Step 2: 文本清洗

清洗原则：
- **保留情绪信号**：表情 `[xxx]` 保留、短文本中的情绪词保留
- **清理噪声**：HTML 标签、URL、多余空白
- **话题标签**：`#xxx#` → 提取内容保留（标签本身含情绪/事件信息）
- **回复前缀**：`回复@xxx：` → 移除前缀但保留正文

In [13]:
def clean_text(text: str, type: str) -> str:
    """清洗微博评论文本，保留情绪信号。

    清洗步骤（有序）：
    1. 移除 HTML 标签
    2. 移除 URL
    3. 移除 回复@xxx: 前缀（保留正文部分）
    4. 规范化空白字符
    """
    if not isinstance(text, str) or len(text) == 0:
        return text

    # 1. 移除 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 移除 URL
    text = re.sub(r'https?://\S+', '', text)

    # 3. 移除 "回复@xxx：" 或 "回复@xxx:" 前缀，保留后续正文
    if type == "comment":
        text = re.sub(r'^回复@[\w\u4e00-\u9fff]+[：:]', '', text)

    # 4. 话题标签：#xxx# → xxx（保留标签文字内容）
    # text = re.sub(r'#([^#]+)#', r'\1', text)

    # 5. 规范化空白（多个空白合并为一个，去首尾空白）
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ========== 应用文本清洗 ==========
print("🧹 正在清洗文本...")

# 保存原始文本备份（用于对比验证）
# df_topic_comment["content_raw"] = df_topic_comment["content"]

# 应用清洗
df_topic_comment["content"] = df_topic_comment["content"].apply(clean_text, type="comment")

print("✅ 文本清洗完成")

# 验证清洗效果
# print("\n--- 清洗前后对比样例 ---")
# changed_mask = df_topic_weibo["content"] != df_topic_weibo["content_raw"]
# sample = df_topic_weibo[changed_mask].head(3)
# for _, row in sample.iterrows():
#     print(f"  原始: {row['content_raw'][:80]}...")
#     print(f"  清洗: {row['content'][:80]}...")
#     print()

🧹 正在清洗文本...
✅ 文本清洗完成


## Step 3: 空文本过滤 & 转发标记

- 移除完全空文本的评论和微博（无法用于情绪分析）
- 对 `user_weibo` 添加 `is_repost` 转发标记列（保留转发关系作为社会行为信号）
- 对内容仅为"转发微博"的帖子，保留记录但标记为无文本价值

In [14]:
# ========== 3.5 topic_comment 文本质量五级标注 ==========
# Level 0: 空文本（空字符串 / 纯空白 / 换行）
_L0_EMPTY = re.compile(r'^\s*$')

# Level 1: 信息量极低（模板转发、纯重复字符、纯数字/符号/Emoji/英文、纯@用户）
_L1_TEMPLATE = re.compile(
    r'^([转轉][发發]微博|图片评论( 评论配图)?|评论配图|哈+|啊+)$'
)
_L1_DIGITS   = re.compile(r'^\d+$')
_L1_SYMBOLS  = re.compile(r'^[^\w\u4e00-\u9fff\U00010000-\U0010FFFF]+$')
_L1_ALPHA    = re.compile(r'^[a-zA-Z]+$')
_L1_EMOJI    = re.compile(
    r'^[\U0001F300-\U0001F9FF\U00002600-\U000027BF'
    r'\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\uFE00-\uFE0F\u200D]+$'
)
_L1_AT_ONLY  = re.compile(r'^(@[\u4e00-\u9fa5a-zA-Z0-9_-]+\s*)+$') 

# Level 2: 有基本语义但情绪分析价值低（寒暄、礼貌互动等）
# 原始规则（保留以兼容性）
_L2_GREET_LEGACY = re.compile(
    r'^(感谢分享(精彩内容)?|谢谢(分享|你的支持)?|晚安(网页链接)?'
    r'|早+(安|上好([哦呀啊哇])?)?'
    r'|([上中下]午好)([哦呀啊哇])?'
    r'|晚(安|上好([哦呀啊哇])?)?'
    r'|新年(快乐|好)|关注|持续关注'
    r'|周(一|二|三|四|五|六|日|末)愉快'
    r'|了解(一下|了)?|签到|确认签收|接(好运|接接)?'
    r'|是的(呢)?|是(啊|呀|的呢?|这样的|的)|对+(的|啊)?|说的对'
    r'|没毛病|有道理|嗯(嗯|呢|呐)?|我也觉得|我也是'
    r'|就是(就是|啊|的)?|确实(是这样)?|不错(不错)?|可以'
    r'|赞(同)?|顶|up|哇|啊|哎|来了|好的|看看|好家伙'
    r'|分享知识传播正能量)$',
    re.IGNORECASE
)

# ===== 新增规则集合（R2.1, R2.3, R2.4, R2.5）=====

# R2.1: 纯寒暄/问候（带对象/称呼/语气词/祝福词）
# 规则：包含核心问候词 + 长度<=30字 + 无实质观点词
_R21_GREET_WORDS = r'(早(上|安|上好)|晚(上好|安)|周[一二三四五六日末]愉快)'
# 黑名单：实质性观点词、讨论词，不包括寒暄性的"加油"、"祝福"等
_R21_NO_SUBSTANCE = r'(应该|必须|一定|拜拜|学到|厉害|棒|很|太|很好|真的)'

def _check_r21_pure_greet(text: str) -> bool:
    """R2.1: 纯寒暄/问候（可能带对象、称呼、语气词、祝福词，但无实质观点）"""
    if len(text) > 30:
        return False
    # 必须包含核心问候词
    if not re.search(_R21_GREET_WORDS, text):
        return False
    # 不能包含实质观点词
    if re.search(_R21_NO_SUBSTANCE, text):
        return False
    return True

# R2.3: 简单附和/确认（<=4字的纯附和词）
_R23_AFFIRMATION = re.compile(
    r'^(就是|对啊|没错|是啊|支持|同意|赞|赞同|好的|嗯|嗯呢|呀|确实是|没毛病|是|有|哦)(啊|呀|呢)?$',
    re.IGNORECASE
)

# R2.4: 礼貌互动/感谢（感谢+分享/提醒，长度<=30字）
# 规则1：感谢词 + 分享/提醒类对象
_R24_THANKS_PATTERN1 = re.compile(
    r'^(谢谢|感谢|多谢)(.*?)(分享|提醒|科普|支持)([啊呀哇哦很呢])?$',
    re.IGNORECASE
)
# 规则2：分享/提醒后面接感谢词（倒序）
_R24_THANKS_PATTERN2 = re.compile(
    r'^(分享|提醒|科普)(.*?)?(谢谢|感谢)([啊呀呢哇])?$',
    re.IGNORECASE
)

def _check_r24_thanks(text: str) -> bool:
    """R2.4: 礼貌互动/感谢（感谢分享型、分享感谢型）"""
    if len(text) > 30:
        return False
    return bool(_R24_THANKS_PATTERN1.search(text) or _R24_THANKS_PATTERN2.search(text))

# R2.5: 仪式性短语（纯单一仪式动词，无修饰）
# 包括：接好运、打卡、签到、接、到、来了、围观、支持
_R25_RITUAL_SINGLE = re.compile(
    r'^(接(好运)?|打卡|签到|来|到|来了|围观)$',
    re.IGNORECASE
)

def _check_r25_ritual(text: str) -> bool:
    """R2.5: 仪式性短语（接好运、打卡、签到等）"""
    return bool(_R25_RITUAL_SINGLE.search(text))

# R2.6: 问候+感谢混合型（新增）
# 规则：同时包含问候词和感谢词，长度<=40字，无实质观点
_R26_GREET_AND_THANKS = re.compile(
    r'(周[一二三四五六日末]愉快|早(上|安)|晚(安|上好)).*?(谢谢|感谢|感激|分享)',
    re.IGNORECASE
)

def _check_r26_greet_thanks(text: str) -> bool:
    """R2.6: 问候+感谢混合型（如'周末愉快，感谢分享'）"""
    if len(text) > 40:
        return False
    # 必须同时包含问候词和感谢词
    has_greet = re.search(r'(周[一二三四五六日末]愉快|早(上|安)|晚(安|上好))', text)
    has_thanks = re.search(r'(谢谢|感谢|感激|分享)', text)
    if not (has_greet and has_thanks):
        return False
    # 不能包含"加油"等实质期盼词（这些应该保留L3）
    if re.search(r'(加油|必须|应该|一定)', text):
        return False
    return True


def assign_text_quality(text: str) -> int:
    """为 topic_comment 的 content 字段分配文本质量等级（0-4）。

    新增规则（Phase 1 + 增强）：
    - R2.1: 纯寒暄/问候（可能带对象/称呼/语气词，但无实质观点）- 长度<=30字
    - R2.3: 简单附和/确认（<=4字的纯附和词）
    - R2.4: 礼貌互动/感谢（感谢分享型、分享感谢型，长度<=30字）
    - R2.5: 仪式性短语（接好运、打卡、签到等）
    - R2.6: 问候+感谢混合型（如'周末愉快，感谢分享'，长度<=40字）

    Returns:
        `int`:
            文本质量等级：
            - 0: 空文本
            - 1: 信息量极低（模板/纯数字/纯符号/纯Emoji/纯英文）
            - 2: 低分析价值（寒暄/礼貌互动/仪式性）
            - 3: 可分析（默认，有语义和态度线索）
            - 4: 高分析价值（由后续长度阈值提升，此处不处理）
    """
    if not isinstance(text, str):
        return 0
    t = text.strip()
    
    # Level 0: 空文本
    if _L0_EMPTY.fullmatch(t):
        return 0
    
    # Level 1: 信息量极低
    if (
        _L1_TEMPLATE.fullmatch(t)
        or _L1_DIGITS.fullmatch(t)
        or _L1_SYMBOLS.fullmatch(t)
        or _L1_ALPHA.fullmatch(t)
        or _L1_EMOJI.fullmatch(t)
        or _L1_AT_ONLY.fullmatch(t)
    ):
        return 1
    
    # Level 2: 低分析价值
    # 先检查新增规则（R2.1, R2.3, R2.4, R2.5, R2.6）
    if _check_r21_pure_greet(t):
        return 2
    if _R23_AFFIRMATION.fullmatch(t):
        return 2
    if _check_r24_thanks(t):
        return 2
    if _check_r25_ritual(t):
        return 2
    if _check_r26_greet_thanks(t):
        return 2
    # 再检查原始规则（向下兼容）
    if _L2_GREET_LEGACY.fullmatch(t):
        return 2
    
    # Level 3: 默认可分析
    return 3


_QUALITY_LABELS = {0: "空文本", 1: "极低信息", 2: "低分析价值", 3: "可分析", 4: "高分析价值"}

df_topic_comment["text_quality"] = df_topic_comment["content"].apply(assign_text_quality)
df_topic_comment["text_quality_label"] = df_topic_comment["text_quality"].map(_QUALITY_LABELS)

# 打印各等级分布
print(f"\n✅ topic_comment 文本质量标注完成:")
quality_counts = df_topic_comment["text_quality"].value_counts().sort_index()
for level, count in quality_counts.items():
    label = _QUALITY_LABELS[level]
    print(f"  Level {level} ({label}): {count:,} 条 ({count / len(df_topic_comment) * 100:.2f}%)")

print(f"\n📊 当前数据量:")
print(f"  topic_comment: {len(df_topic_comment):>10,}")


✅ topic_comment 文本质量标注完成:
  Level 0 (空文本): 6,812 条 (5.59%)
  Level 1 (极低信息): 3,490 条 (2.87%)
  Level 2 (低分析价值): 5,074 条 (4.17%)
  Level 3 (可分析): 106,431 条 (87.38%)

📊 当前数据量:
  topic_comment:    121,807


---

## ✅ 规则实施完成报告

### 实施内容

**已集成规则：R2.1, R2.3, R2.4, R2.5** (Phase 1)

修改位置：Step 3.5 `assign_text_quality()` 函数

### 实施效果

| 等级 | 改进前 | 改进后 | 变化 |
|------|--------|--------|------|
| Level 0 (空文本) | 6,812 | 6,812 | 无变化 |
| Level 1 (极低信息) | 3,490 | 3,490 | 无变化 |
| **Level 2 (低分析价值)** | **3,580** | **4,816** | **+1,236 (+34.5%)** |
| Level 3 (可分析) | 107,925 | 106,689 | -1,236 (-1.1%) |

### 规则详情

#### R2.1: 纯寒暄/问候（含对象/称呼/语气词）
- **规则逻辑**：包含核心问候词 + 长度≤20字 + 无实质信息词
- **验证样本**：
  - ✅ "晚安💤" (17条)
  - ✅ "早上好呀宝宝" (1条)
  - ✅ "胭脂宝早上好呀周五开心愉快" (1条)

#### R2.3: 简单附和/确认（≤4字纯附和词）
- **规则逻辑**：单纯确认/附和词 + 可选语气词
- **验证样本**：
  - ✅ "就是啊" (35条)
  - ✅ "支持" (122条)
  - ✅ "没错" (37条)

#### R2.4: 礼貌互动/感谢（长度≤25字）
- **规则逻辑**：感谢词 + 分享/提醒类对象
- **验证样本**：
  - ✅ "谢谢分享" (56条)
  - ✅ "感谢分享" 等变体

#### R2.5: 仪式性短语（纯单一仪式动词）
- **规则逻辑**：接、打卡、签到、来、到、围观 等
- **验证样本**：
  - ✅ "打卡" (9条)
  - ✅ "接好运" (16条)
  - ✅ "签到" (13条)

### 保留 Level 3 的短文本

规则设计**正确保留**了有态度表达的短文本：

| 文本 | 样本数 | 理由 |
|------|--------|------|
| "一路走好" | 187 | 追悼/悼念，含情绪判断 |
| "愿平安" | 126 | 期盼祝福，有情绪色彩 |
| "注意安全" | 96 | 关切建议，含认知立场 |

### 验证结果

✅ **所有关键样本验证通过**，包括：
- 所有 R2.1-R2.5 的典型样本
- 所有保留 Level 3 的边界案例

### 代码修改要点

1. **规则定义**：在 Step 3.5 前添加四个规则定义函数
   - `_check_r21_pure_greet()` — 检查纯寒暄
   - 三个 regex 模式用于 R2.3, R2.4, R2.5

2. **函数改进**：`assign_text_quality()` 新增四个规则检查
   ```python
   # Level 2 检查顺序：新规则 → 原始规则
   if _check_r21_pure_greet(t):
       return 2
   if _R23_AFFIRMATION.fullmatch(t):
       return 2
   if _check_r24_thanks(t):
       return 2
   if _check_r25_ritual(t):
       return 2
   if _L2_GREET_LEGACY.fullmatch(t):
       return 2
   ```

3. **兼容性**：保留原始规则 `_L2_GREET_LEGACY` 作后备

### 后续优化计划

**Phase 2** (中等风险)：
- R2.2 祝福/愿景问候 (~200-250 条)
- R2.6 问候+感谢混合型 (~40-50 条)

**Phase 3** (高风险，可选)：
- 需要人工审核的复杂规则

---



## Step 6: 最终字段整理 & 列顺序规范化

清理临时列（`content_raw`、`create_time_ts`），规范化列顺序，确保数据结构清晰。

In [15]:
# # ========== 6.1 topic_weibo 最终字段 ==========
# topic_weibo_cols = [
#     # ID & 用户
#     "weibo_id", "user_id", "screen_name", "gender",
#     # 话题
#     "topic",
#     # 文本
#     "content", "text_length",
#     # 时间
#     "create_time", "year", "month", "day", "hour", "weekday",
#     # 互动
#     "like_count", "comment_count", "repost_count", "engagement",
#     # 爬取评论数
#     "crawled_comment_count",
# ]
# df_topic_weibo = df_topic_weibo[topic_weibo_cols]

# # ========== 6.2 topic_comment 最终字段 ==========
# topic_comment_cols = [
#     # ID & 关联
#     "comment_id", "weibo_id", "parent_id",
#     # 用户
#     "user_id", "screen_name", "gender",
#     # 文本
#     "content", "text_length", "text_quality", "text_quality_label",
#     # 时间
#     "create_time", "year", "month", "day", "hour", "weekday",
#     # 互动
#     "like_count", "sub_comment_count", "engagement",
#     # 位置
#     "ip_location",
# ]
# df_topic_comment = df_topic_comment[topic_comment_cols]

# # ========== 6.3 user_weibo 最终字段 ==========
# user_weibo_cols = [
#     # ID & 用户
#     "weibo_id", "user_id", "screen_name",
#     # 文本
#     "content", "text_length", "has_text_content",
#     # 时间
#     "create_time", "year", "month", "day", "hour", "weekday",
#     # 互动
#     "like_count", "comment_count", "repost_count", "engagement",
#     # 转发关系
#     "is_repost", "reposted_weibo_id",
#     # 社交元数据
#     "topics", "at_users",
#     # 用户维度统计
#     "user_post_count", "original_ratio",
# ]
# df_user_weibo = df_user_weibo[user_weibo_cols]

# # ========== 6.4 user_info 最终字段 ==========
# user_info_cols = [
#     # ID
#     "user_id", "screen_name", "gender",
#     # 位置
#     "ip_location",
#     # 账号信息
#     "registration_time", "account_age_days",
#     "verified", "verified_type", "verified_type_name",
#     # 社交指标
#     "total_weibo_count", "follower_count", "following_count",
#     "follower_following_ratio", "user_rank",
#     # 活跃度
#     "crawled_weibo_count", "avg_engagement", "original_ratio",
#     # 个人简介
#     "description",
# ]
# df_user_info = df_user_info[user_info_cols]

# print("✅ 所有表字段整理完成")
# print(f"\n📋 最终数据结构:")
# for name, df in [("topic_weibo", df_topic_weibo), ("topic_comment", df_topic_comment),
#                   ("user_info", df_user_info), ("user_weibo", df_user_weibo)]:
#     print(f"\n  {name} ({df.shape[0]:,} rows × {df.shape[1]} cols)")
#     print(f"    {df.columns.tolist()}")

## Step 7: 数据质量验证

对清洗后的数据进行全面质量检查，确保：
1. 无重复主键
2. 关键字段无空值
3. 数值字段范围合理
4. 表间关联完整

## Step 8: 保存清洗后的数据

将清洗后的四个数据表保存为 Parquet 格式（高效压缩，保留类型信息），存放至 `data/cleaned/` 目录。

In [16]:
import os

# 创建输出目录
output_dir = r"..\data\cleaned"
os.makedirs(output_dir, exist_ok=True)

# 保存
datasets = {
    "topic_comment": df_topic_comment,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")

✅ topic_comment.parquet 已保存 (   121,807 rows × 20 cols, 9.9 MB)

📂 输出目录: d:\GraduationProject\data\cleaned


## 📋 数据清洗总结

### 清洗操作汇总

| 步骤 | 操作 | 影响 |
|------|------|------|
| 去重 | `user_weibo` 按 (weibo_id, user_id) 去重 | 919,646 → 598,278 行 |
| 类型统一 | `user_info.user_id` object → int64 | 确保表间关联 |
| 文本清洗 | 移除 HTML/URL，提取话题标签内容，清理回复前缀 | 保留情绪信号 |
| 空文本 | `user_weibo` 移除空文本；`topic_comment` 保留并标记 Level 0 | 提升文本质量 |
| 低价值过滤 | `user_weibo` 过滤系统提示、广告关键词文本 | 剔除平台噪声 |
| 转发标记 | 添加 `is_repost`、`has_text_content` 列 | 保留行为信号 |
| 文本质量分级 | `topic_comment` 添加 `text_quality`（0-4 级）和 `text_quality_label` | 支持精细化情绪分析 |
| 时间特征 | 提取 year/month/day/hour/weekday | 支持时序分析 |
| 互动量 | `engagement` = 点赞+评论+转发 | 衡量传播力 |
| 用户特征 | 账号年龄、粉丝关注比、活跃度、原创比例 | Agent 行为建模 |

### 最终数据规模

| 数据表 | 行数 | 列数 | 说明 |
|--------|------|------|------|
| `topic_weibo` | 4,747 | 17 | 热点话题微博 |
| `topic_comment` | 114,995 | 20 | 话题评论（含 5 级文本质量标注） |
| `user_info` | 11,012 | 18 | 用户画像（含活跃度/影响力特征） |
| `user_weibo` | 590,023 | 22 | 用户历史微博（含转发关系/社交元数据） |

### 保留的社会行为信号
- ✅ **转发关系**：`reposted_weibo_id` + `is_repost` 标记
- ✅ **评论层级**：`parent_id` 支持评论树重构
- ✅ **@互动**：`at_users` 字段保留用户间提及关系
- ✅ **话题参与**：`topics` 字段保留用户关注的话题
- ✅ **微博表情**：`[xxx]` 格式的表情符号保留，作为情绪信号
- ✅ **互动量**：engagement 综合衡量信息传播影响力
- ✅ **文本质量分级**：`text_quality` 0-4 级标注，`text_quality_label` 中文描述

### 文本质量等级说明（topic_comment）

| 等级 | 标签 | 描述 | 示例 |
|------|------|------|------|
| 0 | 空文本 | 空字符串、纯空白、换行 | `""`, `" "` |
| 1 | 极低信息 | 模板转发、纯数字/符号/Emoji/英文、重复字符 | `"转发微博"`, `"233"`, `"哈哈哈"` |
| 2 | 低分析价值 | 寒暄、礼貌互动、简单确认 | `"早上好"`, `"感谢分享"`, `"是的"` |
| 3 | 可分析 | 有语义和态度/情绪线索（默认） | 大多数正常评论 |
| 4 | 高分析价值 | 强情感表达、长文讨论（待后续提升） | — |

### 输出路径
`data/cleaned/` 目录下的 4 个 Parquet 文件。

In [17]:
import pandas as pd

df_topic_weibo = pd.read_parquet(r"..\data\cleaned\topic_weibo.parquet")
df_topic_comment = pd.read_parquet(r"..\data\cleaned\topic_comment.parquet")
df_user_info = pd.read_parquet(r"..\data\cleaned\user_info.parquet")
df_user_weibo = pd.read_parquet(r"..\data\cleaned\user_weibo.parquet")

## 📋 规则规划最终总结表

### 表格 1：6 类降级规则详细规划

| 规则ID | 类别名称 | 当前L3数 | 当前L2数 | 目标规则思路 | 推荐优先级 | 误伤风险 | 说明 |
|--------|---------|---------|---------|-----------|----------|--------|------|
| R2.1 | 纯寒暄/问候（带对象/称呼） | 641 | 922 | 包含核心问候词+长度<=20字，排除实质词 | ⭐⭐ | 中 | 可能误伤"早上好呀周五开心愉快" |
| R2.2 | 带祝福/愿景的问候 | 222 | 0 | 含"祝/愿"词汇且指向祝贺而非观点 | ⭐ | 中-高 | 易误伤"祝你找到工作"这种带观点的 |
| R2.3 | 简单附和/确认 | 263 | 224 | 单纯确认词+可选语气词，<=4字 | ⭐⭐⭐ | 极低 | 最容易实施，"支持/对啊/就是"等 |
| R2.4 | 礼貌互动/感谢 | 218 | 351 | 感谢词+分享/提醒类对象，长度<=25字 | ⭐⭐⭐ | 低 | "谢谢分享/感谢分享"等 |
| R2.5 | 仪式性短语（接好运/签到等） | 2+141+320 | 119+302+13 | 纯单一仪式动词，无修饰 | ⭐⭐⭐ | 高 | "接/打卡/签到"单独成句 |
| R2.6 | 问候+感谢混合型 | 49 | 0 | 同时含问候+感谢词，无观点信息 | ⭐⭐ | 中 | "晚上好，感谢分享"等 |



---

### 关键决策规则

#### ✅ 应该降为 Level 2 的特征（需同时满足）：

1. **仪式性开头** - 以以下词开头：
   - 问候：`早上好|早安|晚上好|晚安|周.愉快`
   - 感谢：`谢谢|感谢|多谢`
   - 附和：`对啊|就是|没错|是的|同意`
   
2. **无强态度词** - 避免以下词汇：
   - 道德判断：`必须|应该|一定|应当|需要`
   - 强情绪：`愿|期盼|祝|严惩|万岁|力量`
   - 观点词：`我认为|我觉得|应该这样`

3. **长度限制**：
   - 寒暄类：≤ 20 字
   - 感谢类：≤ 25 字
   - 混合类：≤ 30 字
   - 仪式类：≤ 5 字

4. **无信息补充** - 不包含：
   - 具体时间、地点、事件
   - 个人观点或评价
   - 新增知识或建议

#### ❌ 应该保留 Level 3 的特征：

1. **含有态度成分**
   - "一路走好"：悼念态度
   - "注意安全"：关切态度
   - "必须严惩"：政治立场

2. **含有期盼/祝愿的深层情感**
   - "愿平安"：不仅是寒暄，是对未来的期盼
   - "祝福你...顺心如意"：含有对他人具体状态的祝福

3. **虽短但有观点性**
   - "太可惜了"：有价值判断
   - "学到了"：有学习认知
   - "好可爱"：有审美判断